# MGLU3 em 2021 — Análise de Ações com Python

## Pergunta central

**Como o risco, o retorno, o volume e a tendência da MGLU3 evoluíram ao longo de 2021?**

A análise utiliza a série histórica diária do case original. O contexto de mercado de janeiro de 2022 é tratado separadamente e não é usado como prova de causalidade.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/mglu3_2021.csv', parse_dates=['Date'])
df.head()

## 1. Diagnóstico da base

In [ ]:
print('Dimensões:', df.shape)
print('Período:', df['Date'].min(), 'a', df['Date'].max())
print('Valores ausentes:', df.isna().sum().sum())
print('Duplicatas:', df.duplicated().sum())

## 2. Engenharia de métricas

In [ ]:
df = df.sort_values('Date').reset_index(drop=True)
df['daily_return'] = df['Adj Close'].pct_change()
df['cumulative_return'] = df['Adj Close'] / df['Adj Close'].iloc[0] - 1
df['sma_20'] = df['Adj Close'].rolling(20).mean()
df['sma_50'] = df['Adj Close'].rolling(50).mean()
peak = df['Adj Close'].cummax()
df['drawdown'] = df['Adj Close'] / peak - 1
df['rolling_vol_21'] = df['daily_return'].rolling(21).std() * np.sqrt(252)

## 3. Retorno no período

In [ ]:
period_return = df['Adj Close'].iloc[-1] / df['Adj Close'].iloc[0] - 1
print(f'Retorno acumulado: {period_return:.2%}')

## 4. Preço e médias móveis

In [ ]:
ax = df.plot(x='Date', y=['Adj Close', 'sma_20', 'sma_50'], figsize=(12, 6), title='MGLU3 — preço ajustado e médias móveis')
ax.set_ylabel('Preço (R$)')
plt.show()

## 5. Drawdown

In [ ]:
print(f'Drawdown máximo: {df["drawdown"].min():.2%}')
df.plot(x='Date', y='drawdown', figsize=(12, 5), title='Drawdown da MGLU3')
plt.show()

## 6. Volatilidade

In [ ]:
annualized_vol = df['daily_return'].std() * np.sqrt(252)
print(f'Volatilidade anualizada: {annualized_vol:.2%}')
df.plot(x='Date', y='rolling_vol_21', figsize=(12, 5), title='Volatilidade móvel anualizada — 21 pregões')
plt.show()

## 7. Retornos mensais

In [ ]:
monthly_prices = df.set_index('Date')['Adj Close'].resample('ME').last()
monthly = monthly_prices.pct_change()
monthly.iloc[0] = monthly_prices.iloc[0] / df['Adj Close'].iloc[0] - 1
(monthly * 100).round(2)

## 8. Volume e pregões extremos

In [ ]:
best = df.loc[df['daily_return'].idxmax(), ['Date', 'daily_return']]
worst = df.loc[df['daily_return'].idxmin(), ['Date', 'daily_return']]
max_volume = df.loc[df['Volume'].idxmax(), ['Date', 'Volume']]
print('Melhor pregão:\n', best)
print('\nPior pregão:\n', worst)
print('\nMaior volume:\n', max_volume)

## 9. Contexto do case

O material original inclui uma notícia do InfoMoney de 4 de janeiro de 2022 sobre a continuidade da queda de varejistas e menciona um ambiente de juros elevados. Este conteúdo é usado apenas como contexto qualitativo posterior à série de 2021.

A análise não testa causalidade entre juros, notícia e retorno da MGLU3.

## 10. Síntese

- O preço ajustado caiu fortemente ao longo de 2021.
- O retorno acumulado foi de aproximadamente -71,3%.
- O drawdown máximo foi de aproximadamente -78,1%.
- A volatilidade anualizada ficou próxima de 56,9%.
- A deterioração se intensificou no segundo semestre, sobretudo entre setembro e novembro.
- Novembro concentrou tanto o melhor quanto o pior pregão diário do ano, evidenciando alta instabilidade.
- O estudo é descritivo e não constitui recomendação de investimento.